# Placing Xenium cells on their H&E image

The same route as [merfish-visium](merfish-visium.ipynb): the cell cloud is rasterized and
`align_stalign_image` fits image to image, with paired landmarks carrying the correspondence
that intensity alone cannot.

Upstream's equivalent is `xenium-heimage-alignment`, and it runs the pair the other way round --
it warps the H&E onto the rasterized cells and then inverts to place the cells. Here the H&E is
the reference, which is the direction that puts the cells on the image without an inverse.

## Inputs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, spatialdata as sd
from spatialdata.models import Image2DModel, PointsModel
from squidpy.experimental.im import rasterize_points
from squidpy.experimental.tl import align_stalign_image

def as_image(rgb, key):
    return sd.SpatialData(images={key: Image2DModel.parse(
        np.moveaxis(rgb, -1, 0).astype(float), dims=('c', 'y', 'x'))})

def rasterized(xy, dx):
    sdata = sd.SpatialData(points={'cells': PointsModel.parse(xy)})
    rasterize_points(sdata, 'cells', dx=dx, blur=1.0, key_added='section')
    return sdata

he = plt.imread('xenium_data/Xenium_FFPE_Human_Breast_Cancer_Rep1_he_image.png')[..., :3]
cells = pd.read_csv('xenium_data/Xenium_FFPE_Human_Breast_Cancer_Rep1_cells.csv.gz')
xy = np.c_[cells['x_centroid'], cells['y_centroid']].astype(float)

visium = as_image(he, 'he')
xenium = rasterized(xy, 30.0)
print(f'{len(xy)} cells over {xy[:, 0].max():.0f} x {xy[:, 1].max():.0f} um, '
      f'rasterized to {tuple(np.asarray(xenium["section"]).shape)}; H&E is {he.shape}')

Four landmark pairs, hardcoded upstream. They are stored there as `(y, x)` -- squidpy's public
API takes `(x, y)`, so each is reversed once on the way in rather than the arrays being
transposed later.

In [ ]:
landmarks_he = np.array([[1050., 950.], [700., 2200.], [500., 1550.], [1550., 1840.]])[:, ::-1]
landmarks_cells = np.array([[3108., 2100.], [4480., 6440.], [5040., 4200.], [1260., 5320.]])[:, ::-1]

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
ax[0].imshow(he); ax[0].scatter(*landmarks_he.T, s=12, c='red')
ax[0].set_title('H&E, in pixels')
ax[1].scatter(*xy.T, s=0.12, alpha=0.3); ax[1].scatter(*landmarks_cells.T, s=12, c='red')
ax[1].set_title('Xenium cells, in microns'); ax[1].invert_yaxis(); ax[1].set_aspect('equal')
for a in ax:
    a.set_xticks([]); a.set_yticks([])

## The fit

Upstream's own solver values for this pair.

In [ ]:
fit = align_stalign_image(
    visium, xenium, image_key=('he', 'section'),
    landmarks_ref=landmarks_he, landmarks_query=landmarks_cells,
    niter=2000, sigmaM=0.15, sigmaB=0.10, sigmaA=0.11, epV=10,
)
print(f'{fit.n_iter} iterations, objective '
      f'{float(fit.energies[0]):.0f} -> {float(fit.energies[-1]):.0f}')

## Every cell, placed on the image

In [ ]:
placed = np.asarray(fit.transform(xy))
residual = np.linalg.norm(np.asarray(fit.transform(landmarks_cells)) - landmarks_he, axis=1)
rows, columns = he.shape[:2]
inside = ((placed[:, 0] >= 0) & (placed[:, 0] < columns)
          & (placed[:, 1] >= 0) & (placed[:, 1] < rows))
print(f'landmark residual: median {np.median(residual):.1f} px, worst {residual.max():.1f} px')
print(f'{100 * inside.mean():.0f}% of cells land within the {columns} x {rows} image')

fig, ax = plt.subplots(1, 2, figsize=(13, 6))
ax[0].imshow(he); ax[0].set_title('H&E')
ax[1].imshow(he); ax[1].scatter(*placed.T, s=0.12, alpha=0.3, c='tab:blue')
ax[1].scatter(*landmarks_he.T, s=12, c='red', label='target landmarks')
ax[1].set_title('Xenium cells placed on it'); ax[1].legend(fontsize=8)
for a in ax:
    a.set_xticks([]); a.set_yticks([])

## The objective's trace

In [ ]:
MIXTURE_GATE = 50
energies = np.asarray(fit.energies)[: fit.n_iter]
descent = energies[MIXTURE_GATE + 1 :]
tail = descent[-max(len(descent) // 10, 1) :]
print(f'after the gate: {descent[0]:.0f} -> {descent[-1]:.0f}, minimum {descent.min():.0f} '
      f'at iteration {MIXTURE_GATE + 1 + int(descent.argmin())}')
print(f'last tenth: mean {tail.mean():.0f}, spread {np.ptp(tail):.0f} '
      f'({100 * np.ptp(tail) / tail.mean():.1f}% of its mean)')
plt.plot(energies, lw=0.8); plt.axvline(MIXTURE_GATE, color='0.6', ls='--', lw=0.8)
plt.xlabel('iteration'); plt.ylabel('objective'); plt.grid(alpha=0.3)